# Path Classification

In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## 1. Dataset Creation

In [ ]:
import os
import shutil
import glob

# Raw image --> Dataset
raw_img_path = "/content/gdrive/MyDrive/leepi/raw/250202"
dataset_path = "/content/gdrive/MyDrive/leepi/dataset/v0"

# Dataset Path by Class
dest_dirs = {"GO": os.path.join(dataset_path, "front"),
             "LEFT": os.path.join(dataset_path, "left"),
             "RIGHT": os.path.join(dataset_path, "right"),}

# Dataset Folder Create
os.makedirs(dataset_path, exist_ok=True)
for path in dest_dirs.values():
    os.makedirs(path, exist_ok=True)

# Search .jpg file & Copy to destination
for img_file in glob.glob(os.path.join(raw_img_path, "*.jpg")):
    _class = img_file.rsplit('_', 1)[-1][:-4]
    if _class in dest_dirs:
        shutil.copy(img_file, os.path.join(dest_dirs[_class], os.path.basename(img_file)))

print("Dataset Created!!")

Dataset Created!!


# 2. Dataset PreProcessing

In [2]:
transform = transforms.Compose([transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

# 모든 데이터를 한 번에 로드
dataset_path = "/content/gdrive/MyDrive/leepi/dataset/v0"

dataset = ImageFolder(root=dataset_path, transform=transform)

# 클래스 목록 확인
num_classes = len(dataset.classes)

# train / test split
train_indices, test_indices = train_test_split(np.arange(len(dataset)), test_size=0.2, stratify=dataset.targets, random_state=42)

# Subset으로 나누기
train_dataset = Subset(dataset, train_indices)
test_dataset  = Subset(dataset, test_indices)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Total Dataset : {len(dataset)} (Train : {len(train_dataset)} / Test : {len(test_dataset)})")
print(f"Class List : {dataset.classes}\n")

Total Dataset : 203 (Train : 162 / Test : 41)
Class List : ['front', 'left', 'right']



# 3. Model Customization

In [10]:
from torchvision.models import efficientnet_b0
import torch.nn as nn

# 모델 불러오기
model = efficientnet_b0(weights='DEFAULT')
# model = efficientnet_b0(weights='IMAGENET1K_V1')

# 출력 레이어 수정 (3개 클래스 분류)
model.classifier[1] = nn.Linear(in_features=1280, out_features=num_classes)

# 모델을 GPU로 이동 (가능한 경우)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.train()
model.qconfig = torch.quantization.get_default_qconfig('qnnpack')  # 기본 QAT 설정

torch.backends.quantized.engine = 'qnnpack'
torch.quantization.prepare_qat(model, inplace=True)

print(f"Model Customization Complete!! (device : {device})")

Model Customization Complete!! (device : cuda)


# 4. Training

In [4]:
import torch.optim as optim

# 손실 함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 함수 정의
def train(model, train_loader, test_loader, epochs=5):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct, total = 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {accuracy:.2f}%")

    print("학습 완료!")

# 학습 실행
train(model, train_loader, test_loader, epochs=10)

Epoch [1/10], Loss: 0.8064, Accuracy: 66.05%
Epoch [2/10], Loss: 0.4955, Accuracy: 82.72%
Epoch [3/10], Loss: 0.2561, Accuracy: 89.51%
Epoch [4/10], Loss: 0.2450, Accuracy: 91.98%
Epoch [5/10], Loss: 0.3658, Accuracy: 95.06%
Epoch [6/10], Loss: 0.2201, Accuracy: 91.36%
Epoch [7/10], Loss: 0.5642, Accuracy: 91.36%
Epoch [8/10], Loss: 0.2857, Accuracy: 93.21%
Epoch [9/10], Loss: 0.2272, Accuracy: 88.89%
Epoch [10/10], Loss: 0.2411, Accuracy: 93.83%
학습 완료!
모델 저장 완료


# 5. Quantization

In [14]:
import os

weight_path = "/content/gdrive/MyDrive/leepi/weight"
weight_name = "effnetlite_250204"

# 모델 pth 저장

torch.save(model.state_dict(), "efficientnet_lite_3class.pth")
print("모델 저장 완료")


# 모델을 양자화
model.cpu()
model.eval()

# 양자화 적용
quantized_model = torch.quantization.convert(model, inplace=False)

# TorchScript 변환
scripted_model = torch.jit.script(quantized_model)
scripted_model.save(os.path.join(weight_path, weight_name + "_scripted.pt"))

In [17]:
!pip install torchinfo

In [27]:
from torchinfo import summary

input_size = (1, 3, 224, 224)  # 예시: 배치 사이즈 1, 3채널, 224x224 이미지

print("=== Original Model Summary ===")
# summary(model, input_size=input_size)
print(model.state_dict().keys())

# print("\n=== Quantized Model Summary ===")
print(quantized_model.state_dict().keys())
# summary(quantized_model, input_size=input_size, device="cpu")



=== Original Model Summary ===
odict_keys(['features.0.0.weight', 'features.0.0.weight_fake_quant.eps', 'features.0.0.weight_fake_quant.min_val', 'features.0.0.weight_fake_quant.max_val', 'features.0.0.activation_post_process.eps', 'features.0.0.activation_post_process.histogram', 'features.0.0.activation_post_process.min_val', 'features.0.0.activation_post_process.max_val', 'features.0.1.weight', 'features.0.1.bias', 'features.0.1.running_mean', 'features.0.1.running_var', 'features.0.1.num_batches_tracked', 'features.0.1.activation_post_process.eps', 'features.0.1.activation_post_process.histogram', 'features.0.1.activation_post_process.min_val', 'features.0.1.activation_post_process.max_val', 'features.1.0.block.0.0.weight', 'features.1.0.block.0.0.weight_fake_quant.eps', 'features.1.0.block.0.0.weight_fake_quant.min_val', 'features.1.0.block.0.0.weight_fake_quant.max_val', 'features.1.0.block.0.0.activation_post_process.eps', 'features.1.0.block.0.0.activation_post_process.histogra

In [29]:
def print_module_summary(module, prefix=""):
    for name, child in module.named_children():
        child_str = str(child)
        num_params = sum(p.numel() for p in child.parameters())
        print(f"{prefix}{name}: {child_str}  [Params: {num_params}]")
        #print_module_summary(child, prefix=prefix + "  ")

print("=== Original Model Summary ===")
print_module_summary(model)

print("=== Quantized Model Summary (Custom) ===")
print_module_summary(quantized_model)


=== Original Model Summary ===
features: Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(
      3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
      (weight_fake_quant): MinMaxObserver(min_val=-2.5309460163116455, max_val=2.3643903732299805)
      (activation_post_process): HistogramObserver(min_val=-7.619816303253174, max_val=5.753482341766357)
    )
    (1): BatchNorm2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (activation_post_process): HistogramObserver(min_val=-18.953372955322266, max_val=16.933515548706055)
    )
    (2): SiLU(inplace=True)
  )
  (1): Sequential(
    (0): MBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(
            32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False
            (weight_fake_quant): MinMaxObserver(min_val=-2.492830753326416, max_val=3.63584303855896)
            (activation_post_process): HistogramObserver(m

In [ ]:
import torch.quantization

weight_path = "/content/gdrive/MyDrive/leepi/weight"
weight_name = "effnetlite_250204"

# 모델을 평가 모드로 변경
model.eval()


import torch
import torch.nn as nn

# Conv2dNormActivation 클래스를 직접 퓨징하는 함수
def fuse_conv_bn_silu(model):
    # 모델의 각 계층을 순차적으로 확인하며, Conv2d + BatchNorm + SiLU를 퓨즈
    for name, module in model.named_children():
        if isinstance(module, nn.Sequential):
            # Conv2d, BatchNorm2d, SiLU가 순차적으로 나열된 계층에서 퓨즈
            if len(module) == 3 and isinstance(module[0], nn.Conv2d) and isinstance(module[1], nn.BatchNorm2d) and isinstance(module[2], nn.SiLU):
                # Conv2d + BatchNorm + SiLU를 하나의 Conv2d로 퓨즈
                fused_conv = torch.nn.Sequential(
                    nn.Conv2d(module[0].in_channels, module[0].out_channels, module[0].kernel_size,
                              module[0].stride, module[0].padding, bias=False),
                    nn.BatchNorm2d(module[1].num_features),
                    nn.SiLU()
                )
                setattr(model, name, fused_conv)
    return model

# EfficientNet 모델의 특정 부분을 퓨징
fused_model = fuse_conv_bn_silu(model)


# 일부 계층만 양자화 적용 가능하므로, Fused 모델 생성
# fused_model = torch.quantization.fuse_modules(model, [["features.0", "features.1", "features.2"]])

# 양자화 적용 (Dynamic Quantization)
quantized_model = torch.quantization.quantize_dynamic(fused_model, {torch.nn.Linear}, dtype=torch.qint8)
torch.save(quantized_model.state_dict(), os.path.join(weight_path, weight_name + ".pth"))

# TorchScript 변환
scripted_model = torch.jit.script(quantized_model)
scripted_model.save(os.path.join(weight_path, weight_name + "_scripted.pt"))

print("라즈베리파이에서 실행 가능하도록 모델 저장 완료!")

RuntimeError: false INTERNAL ASSERT FAILED at "../aten/src/ATen/quantized/Quantizer.cpp":445, please report a bug to PyTorch. cannot call qscheme on UnknownQuantizer

In [ ]:
import torch
import torch.nn as nn
import os

# Define your model (assuming it's already defined)
# model = ...

# Fuse Conv2d + BatchNorm + SiLU layers
def fuse_conv_bn_silu(model):
    for name, module in model.named_children():
        if isinstance(module, nn.Sequential):
            if len(module) == 3 and isinstance(module[0], nn.Conv2d) and isinstance(module[1], nn.BatchNorm2d) and isinstance(module[2], nn.SiLU):
                fused_conv = torch.nn.Sequential(
                    nn.Conv2d(module[0].in_channels, module[0].out_channels, module[0].kernel_size,
                              module[0].stride, module[0].padding, bias=False),
                    nn.BatchNorm2d(module[1].num_features),
                    nn.SiLU()
                )
                setattr(model, name, fused_conv)
    return model

# Fuse the model
fused_model = fuse_conv_bn_silu(model)

# Print the fused model architecture for verification
print(fused_model)

# Apply dynamic quantization
quantized_model = torch.quantization.quantize_dynamic(fused_model, {torch.nn.Linear, torch.nn.Conv2d}, dtype=torch.qint8)


# Print the quantized model architecture for verification
print(quantized_model)

# Save the quantized model
weight_path = "/content/gdrive/MyDrive/leepi/weight"
weight_name = "effnetlite_250204"
torch.save(quantized_model.state_dict(), os.path.join(weight_path, weight_name + ".pth"))

# Convert to TorchScript
try:
    scripted_model = torch.jit.script(quantized_model)
    scripted_model.save(os.path.join(weight_path, weight_name + "_scripted.pt"))
    print("라즈베리파이에서 실행 가능하도록 모델 저장 완료!")
except Exception as e:
    print(f"TorchScript 변환 중 오류 발생: {e}")

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

# 라즈베리파에서

In [ ]:
import torch

model = torch.jit.load("efficientnet_lite_3class_quantized_scripted.pt")
model.eval()

# 정적 양자화

In [ ]:
# __init__(self) 메소드에 삽입
# self.quant = torch.quantization.QuantStub()
# self.dequant = torch.quantization.DeQuantStub()



backend = "qnnpack"
model.qconfig = torch.quantization.get_default_qconfig(backend)
torch.backends.quantized.engine = backend
model_quantized_static = torch.quantization.prepare(model, inplace=False)
model_quantized_static = torch.quantization.convert(model_quantized_static, inplace=False)